In [1]:
from pyspark.sql import SparkSession

# packages for iceberg + spark + kafka
PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Iceberg_Setup") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [5]:
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, from_json

# configurations
intervals = [30,60] # compaction intervals in minutes
run_duration_sec = 5 * 60 * 60 # 2 hours per phase
table_name = "my_catalog.default.user_events_time_based_partitioned"

# json schema definition
json_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("shop", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

def clean_environment(phase_name):
    # drop the table if it exists
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    
    # recreate the clean table structure
    spark.sql(f"""
    CREATE TABLE {table_name} (
        user_id INT,
        shop STRING,    
        event_type STRING,
        device_type STRING,
        price DOUBLE,
        session_id STRING,
        timestamp TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (hours(timestamp))
    TBLPROPERTIES (
        'format-version'='2',
        'write.format.default'='parquet'
    )
    """)
    print(f"[{datetime.now()}] environment cleaned for phase: {phase_name}")

# main orchestration loop across all intervals
for interval in intervals:
    phase_label = f"{interval}min_interval"
    print(f"\n================ starting phase: {phase_label} ================")
    
    # step a: clean metadata and tables to prevent benchmark bleed
    clean_environment(phase_label)
    
    # step b: launch streaming query with isolated and unique checkpoint folder
    unique_run_id = int(time.time())
    checkpoint_path = f"s3a://warehouse/checkpoints/time_based_{phase_label}_{unique_run_id}"
    
    kafka_stream = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka:9092") \
        .option("subscribe", "user_events") \
        .option("startingOffsets", "latest") \
        .load()

    parsed_stream = kafka_stream \
        .selectExpr("CAST(value AS STRING) as json_string") \
        .select(from_json(col("json_string"), json_schema).alias("data")) \
        .select("data.*")
    
    streaming_query = parsed_stream.writeStream \
        .format("iceberg") \
        .outputMode("append") \
        .trigger(processingTime="5 seconds") \
        .option("checkpointLocation", checkpoint_path) \
        .toTable(table_name)
        
    print(f"stream active for {phase_label}. beginning execution timer.")
    
    # step c: monitor and execute compaction on schedule
    start_phase_time = time.time()
    last_compaction_time = time.time()
    
    while time.time() - start_phase_time < run_duration_sec:
        # check if it is time to run the periodic z-order compaction
        if time.time() - last_compaction_time >= (interval * 60):
            print(f"[{datetime.now()}] triggering scheduled compaction via spark internal call...")
            
            compaction_start = time.time()
            
            # call optimization directly without conflicting with active ingestion stream
            spark.sql(f"""
                CALL my_catalog.system.rewrite_data_files(
                    table => '{table_name.replace("my_catalog.", "")}',
                    strategy => 'sort',
                    sort_order => 'zorder(shop, device_type, event_type)',
                    where => 'timestamp < current_timestamp() - INTERVAL 5 MINUTE',
                    options => map(
                        'partial-progress.enabled', 'true',
                        'partial-progress.max-commits', '10',
                        'min-input-files', '10'
                    )
                )
            """)
            
            compaction_duration = time.time() - compaction_start
            last_compaction_time = time.time()
            
            print(f"[{datetime.now()}] compaction cycle complete in {compaction_duration:.2f} seconds.")
            
        time.sleep(10) # sleep loop check
        
    # step d: graceful shutdown of active stream before moving to next phase
    print(f"phase {phase_label} completed. terminating stream cleanly...")
    streaming_query.stop()
    streaming_query.awaitTermination()

print("all automated benchmarking phases executed successfully.")


================ starting phase: 30min_interval ================
[2026-05-22 18:16:27.879144] environment cleaned for phase: 30min_interval
stream active for 30min_interval. beginning execution timer.
[2026-05-22 18:46:28.122164] triggering scheduled compaction via spark internal call...
[2026-05-22 18:46:50.699053] compaction cycle complete in 22.58 seconds.
[2026-05-22 19:16:50.719217] triggering scheduled compaction via spark internal call...
[2026-05-22 19:17:25.288008] compaction cycle complete in 34.57 seconds.
[2026-05-22 19:47:25.793525] triggering scheduled compaction via spark internal call...
[2026-05-22 19:47:51.145173] compaction cycle complete in 25.35 seconds.
[2026-05-22 20:17:51.165425] triggering scheduled compaction via spark internal call...
[2026-05-22 20:18:19.157469] compaction cycle complete in 27.99 seconds.
[2026-05-22 20:48:19.182982] triggering scheduled compaction via spark internal call...
[2026-05-22 20:48:46.610121] compaction cycle complete in 27.43 se

In [3]:
spark.sql(f"DROP TABLE IF EXISTS my_catalog.default.user_events_time_based PURGE")

DataFrame[]

In [ ]:
spark.sql("DESCRIBE my_catalog.default.user_events_time_based").show()

In [ ]:
================ starting phase: 10min_interval ================
[2026-05-22 13:17:53.644612] environment cleaned for phase: 10min_interval
stream active for 10min_interval. beginning execution timer.
[2026-05-22 13:27:54.104594] triggering scheduled compaction via spark internal call...
[2026-05-22 13:28:07.072047] compaction cycle complete in 12.97 seconds.
[2026-05-22 13:38:07.078699] triggering scheduled compaction via spark internal call...
[2026-05-22 13:38:16.699942] compaction cycle complete in 9.62 seconds.
[2026-05-22 13:48:16.707093] triggering scheduled compaction via spark internal call...
[2026-05-22 13:48:29.395565] compaction cycle complete in 12.69 seconds.
[2026-05-22 13:58:29.402829] triggering scheduled compaction via spark internal call...
[2026-05-22 13:58:43.953215] compaction cycle complete in 14.55 seconds.
[2026-05-22 14:08:43.960011] triggering scheduled compaction via spark internal call...
[2026-05-22 14:08:59.331088] compaction cycle complete in 15.37 seconds.
[2026-05-22 14:18:59.337565] triggering scheduled compaction via spark internal call...
[2026-05-22 14:19:07.659427] compaction cycle complete in 8.32 seconds.
[2026-05-22 14:29:07.665948] triggering scheduled compaction via spark internal call...
[2026-05-22 14:29:20.547700] compaction cycle complete in 12.88 seconds.
[2026-05-22 14:39:20.558130] triggering scheduled compaction via spark internal call...
[2026-05-22 14:39:34.947486] compaction cycle complete in 14.39 seconds.
[2026-05-22 14:49:34.954165] triggering scheduled compaction via spark internal call...
[2026-05-22 14:49:50.053479] compaction cycle complete in 15.10 seconds.
[2026-05-22 14:59:50.066525] triggering scheduled compaction via spark internal call...
[2026-05-22 15:00:09.357971] compaction cycle complete in 19.29 seconds.
[2026-05-22 15:10:09.365083] triggering scheduled compaction via spark internal call...
[2026-05-22 15:10:25.007206] compaction cycle complete in 15.64 seconds.
[2026-05-22 15:20:25.013806] triggering scheduled compaction via spark internal call...
[2026-05-22 15:20:36.144807] compaction cycle complete in 11.13 seconds.
[2026-05-22 15:30:36.157530] triggering scheduled compaction via spark internal call...
[2026-05-22 15:30:47.317487] compaction cycle complete in 11.16 seconds.
[2026-05-22 15:40:47.323867] triggering scheduled compaction via spark internal call...
[2026-05-22 15:40:56.964354] compaction cycle complete in 9.64 seconds.
[2026-05-22 15:50:56.971165] triggering scheduled compaction via spark internal call...
[2026-05-22 15:51:09.870819] compaction cycle complete in 12.90 seconds.
[2026-05-22 16:01:09.880389] triggering scheduled compaction via spark internal call...
[2026-05-22 16:01:24.353277] compaction cycle complete in 14.47 seconds.
[2026-05-22 16:11:24.359885] triggering scheduled compaction via spark internal call...
[2026-05-22 16:11:37.206794] compaction cycle complete in 12.85 seconds.
[2026-05-22 16:21:37.213260] triggering scheduled compaction via spark internal call...
[2026-05-22 16:21:44.397298] compaction cycle complete in 7.18 seconds.
[2026-05-22 16:31:44.403874] triggering scheduled compaction via spark internal call...
[2026-05-22 16:31:54.723994] compaction cycle complete in 10.32 seconds.
[2026-05-22 16:41:54.730713] triggering scheduled compaction via spark internal call...
[2026-05-22 16:42:05.719535] compaction cycle complete in 10.99 seconds.
[2026-05-22 16:52:05.725912] triggering scheduled compaction via spark internal call...
[2026-05-22 16:52:19.062777] compaction cycle complete in 13.34 seconds.
[2026-05-22 17:02:19.069731] triggering scheduled compaction via spark internal call...
[2026-05-22 17:02:33.861069] compaction cycle complete in 14.79 seconds.
[2026-05-22 17:12:33.867945] triggering scheduled compaction via spark internal call...
[2026-05-22 17:12:46.639079] compaction cycle complete in 12.77 seconds.
[2026-05-22 17:22:46.645540] triggering scheduled compaction via spark internal call...
[2026-05-22 17:22:55.520235] compaction cycle complete in 8.87 seconds.
[2026-05-22 17:32:55.526670] triggering scheduled compaction via spark internal call...
[2026-05-22 17:33:04.671815] compaction cycle complete in 9.14 seconds.
[2026-05-22 17:43:04.678250] triggering scheduled compaction via spark internal call...
[2026-05-22 17:43:15.246682] compaction cycle complete in 10.57 seconds.
[2026-05-22 17:53:15.277065] triggering scheduled compaction via spark internal call...
[2026-05-22 17:53:29.029287] compaction cycle complete in 13.75 seconds.
[2026-05-22 18:03:29.036203] triggering scheduled compaction via spark internal call...
[2026-05-22 18:03:44.481255] compaction cycle complete in 15.44 seconds.
[2026-05-22 18:13:44.488129] triggering scheduled compaction via spark internal call...
[2026-05-22 18:13:57.366180] compaction cycle complete in 12.88 seconds.

In [ ]:
================ starting phase: 30min_interval ================
[2026-05-22 18:16:27.879144] environment cleaned for phase: 30min_interval
stream active for 30min_interval. beginning execution timer.
[2026-05-22 18:46:28.122164] triggering scheduled compaction via spark internal call...
[2026-05-22 18:46:50.699053] compaction cycle complete in 22.58 seconds.
[2026-05-22 19:16:50.719217] triggering scheduled compaction via spark internal call...
[2026-05-22 19:17:25.288008] compaction cycle complete in 34.57 seconds.
[2026-05-22 19:47:25.793525] triggering scheduled compaction via spark internal call...
[2026-05-22 19:47:51.145173] compaction cycle complete in 25.35 seconds.
[2026-05-22 20:17:51.165425] triggering scheduled compaction via spark internal call...
[2026-05-22 20:18:19.157469] compaction cycle complete in 27.99 seconds.
[2026-05-22 20:48:19.182982] triggering scheduled compaction via spark internal call...
[2026-05-22 20:48:46.610121] compaction cycle complete in 27.43 seconds.
[2026-05-22 21:18:46.630960] triggering scheduled compaction via spark internal call...
[2026-05-22 21:19:02.384734] compaction cycle complete in 15.75 seconds.
[2026-05-22 21:49:02.404971] triggering scheduled compaction via spark internal call...
[2026-05-22 21:49:33.256185] compaction cycle complete in 30.85 seconds.
[2026-05-22 22:19:33.275240] triggering scheduled compaction via spark internal call...
[2026-05-22 22:19:53.239358] compaction cycle complete in 19.96 seconds.
[2026-05-22 22:49:53.259535] triggering scheduled compaction via spark internal call...
[2026-05-22 22:50:23.086583] compaction cycle complete in 29.83 seconds.
phase 30min_interval completed. terminating stream cleanly...

================ starting phase: 60min_interval ================
[2026-05-22 23:16:33.188391] environment cleaned for phase: 60min_interval
stream active for 60min_interval. beginning execution timer.
[2026-05-23 00:16:33.435466] triggering scheduled compaction via spark internal call...
[2026-05-23 00:17:44.408283] compaction cycle complete in 70.97 seconds.
[2026-05-23 01:17:44.464149] triggering scheduled compaction via spark internal call...
[2026-05-23 01:19:09.178523] compaction cycle complete in 84.71 seconds.
[2026-05-23 02:19:09.219938] triggering scheduled compaction via spark internal call...
[2026-05-23 02:20:25.884928] compaction cycle complete in 76.66 seconds.
[2026-05-23 03:20:32.577064] triggering scheduled compaction via spark internal call...
[2026-05-23 03:21:59.837220] compaction cycle complete in 87.26 seconds.
phase 60min_interval completed. terminating stream cleanly...
all automated benchmarking phases executed successfully.